# Microglia Heatmap-based Localization

This notebook contains the code to execute and evaluate microglia localization utilizing heatmaps generated by a UNet machine learning model. The heatmaps correspond to likely regions where a cell soma can be found.

## Training Loop

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from scripts.filters import *
from scripts.utils import *
import albumentations as A
from torch.utils.data import Dataset, DataLoader, random_split
from albumentations.pytorch import ToTensorV2
from torchvision import transforms
import torch 
from torch.utils.data import Subset
import torch.nn as nn
import segmentation_models_pytorch as smp
from tqdm import tqdm
import gc 


class MicrogliaHeatmapDataset(Dataset):

    def __init__(self, image_dir, json_path, sigma=20.0, augment=False):
        self.image_dir = Path(image_dir)
        self.annotations = extract_points_from_json(json_path)
        self.file_names = [f.name for f in self.image_dir.glob("*.png")]
        self.sigma = sigma
        self.augment = augment
        self.train_aug = A.Compose(
            [
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.RandomBrightnessContrast(p=0.3),
                A.GaussNoise(p=0.2),
                A.GaussianBlur(blur_limit=3, p=0.1),
                A.Normalize(),
                ToTensorV2(),
            ]
        )
        self.val_aug = A.Compose(
            [
                A.Normalize(),
                ToTensorV2(),
            ]
        )
        self.heatmaps = {}
        for fname in self.file_names:
            with Image.open(self.image_dir / fname) as img:
                w, h = img.size
            pts = self.annotations.get(fname, [])
            self.heatmaps[fname] = make_gaussian_heatmap(
                (h, w), pts, sigma=self.sigma
            ).astype(np.float32)
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        fname = self.file_names[idx]
        with Image.open(self.image_dir / fname) as img:
            img = np.array(img.convert("RGB"))

        aug = self.train_aug if self.augment else self.val_aug
        result = aug(image=img, mask=self.heatmaps[fname])
        x = result["image"].float()
        y = result["mask"].unsqueeze(0).float()
        return x, y


dataset_split = "train"
IMAGE_DIR = f"./evaluation/{dataset_split}"
JSON_PATH = "./evaluation/annotations/gt_points.json"
SAVE_PATH = "./evaluation/checkpoints/best_model.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4
NUM_EPOCHS = 75
LR = 1e-4
VAL_SPLIT = 0.2
SIGMA = 20.0

train_dataset = MicrogliaHeatmapDataset(IMAGE_DIR, JSON_PATH, sigma=SIGMA, augment=True)
full_dataset = MicrogliaHeatmapDataset(IMAGE_DIR, JSON_PATH, sigma=SIGMA, augment=False)

val_size = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
generator = torch.Generator().manual_seed(30)
train_indices, val_indices = random_split(
    range(len(full_dataset)), [train_size, val_size], generator=generator
)


train_dataset = Subset(train_dataset, train_indices.indices)
val_dataset = Subset(full_dataset, val_indices.indices)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

model = smp.Unet().to(DEVICE)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5
)

best_val_loss = float("inf")


for epoch in tqdm(range(NUM_EPOCHS)):

    model.train()
    train_loss = 0.0
    for step, (images, heatmaps) in enumerate(train_loader):
        images = images.to(DEVICE)
        heatmaps = heatmaps.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        preds = torch.sigmoid(logits)
        loss = loss_fn(preds, heatmaps)

        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        del images, heatmaps, logits, preds, loss

    train_loss /= len(train_loader)
    model.eval()
    val_loss = 0.0

    with torch.inference_mode():
        for images, heatmaps in val_loader:
            images = images.to(DEVICE)
            heatmaps = heatmaps.to(DEVICE)
            preds = torch.sigmoid(model(images))
            val_loss += loss_fn(preds, heatmaps).item()

    val_loss /= len(val_loader)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
            },
            SAVE_PATH,
        )
        print(f"Saved best model: {val_loss:.4f}")

    gc.collect()

print("Training complete. Best val loss:", best_val_loss)

## Full Pipeline for Bounding Box Calculation

In [ ]:
from pathlib import Path

path = "/media/sdog/SSD-500/microglia_data"
files = [f for f in Path(path).glob("./*.tiff")]
classes = list(map((lambda x: 0 if "EV" in x else 1), [f.name for f in files]))

print(files)

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from scripts.filters import *
from scripts.evaluate import *
from scripts.utils import *
from torch.utils.data import IterableDataset
import pyvips
from scipy import ndimage as ndi
from skimage.segmentation import watershed
import pandas as pd
from sklearn.cluster import DBSCAN
from scipy.ndimage import label, center_of_mass, maximum_filter


class TileDataset(IterableDataset):
    def __init__(self, file, label, patch_size=512, transform=None):
        super().__init__()
        self.file = file
        self.patch_size = patch_size
        self.transform = transform
        self.label = label

    def __iter__(self):
        img = pyvips.Image.new_from_file(self.file, access="sequential")
        for _, patch_np, (y, x) in create_patches(
            img, 512, stride=512, filter=filter_tissue
        ):
            if patch_np.shape[2] == 4:
                patch_np = patch_np[:, :, :3]
            if self.transform:
                patch_np = self.transform(patch_np)
            patch_tensor = torch.from_numpy(patch_np).permute(2, 0, 1).float() / 255.0
            yield patch_tensor, self.label, y, x


def extract_cell_coordinates(heatmap, threshold=0.7, min_distance=20):
    """Given a heatmap, extract potential cell coordinates by finding local maxima above a threshold"""
    local_max = heatmap == maximum_filter(heatmap, size=min_distance)
    local_max = local_max & (heatmap > threshold)

    labeled, num_cells = label(local_max)
    centroids = center_of_mass(heatmap, labeled, range(1, num_cells + 1))

    return centroids, num_cells


def merge_close_centroids(centroids, distance_threshold=30):
    """Given potential cell centroids, merge those that are within the specified distance threshold into a single centroid using mean location"""
    if len(centroids) == 0:
        return []

    pts = np.array(centroids)
    labels = DBSCAN(eps=distance_threshold, min_samples=1).fit_predict(pts)

    merged = []
    for lab in np.unique(labels):
        cluster_pts = pts[labels == lab]
        merged_centroid = cluster_pts.mean(axis=0)
        merged.append(tuple(merged_centroid))

    return merged



def binarize_HSV(img_np):
    """Binarize image patch for branch extraction and bounding box calculation"""

    # HSV color segmentation:
    img_hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)

    lb = np.array([0, 0, 0])
    ub = np.array([15, 255, 255])
    mask = cv2.inRange(img_hsv, lb, ub)

    # morphology to connect branches
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)



BATCH_SIZE = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LR = 1e-4
margin = 50
model = smp.Unet().to(DEVICE)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5
)
checkpoint = torch.load("./checkpoints/resnet18_sigma15_aug.pth", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

for idx, file in enumerate(files):

    dataset = TileDataset(file, classes[idx])
    loader = DataLoader(dataset, BATCH_SIZE, num_workers=0)

    all_records = []
    vips_img = pyvips.Image.new_from_file(str(file), access="random")
    tile_x = 0
    tile_y = 0
    tiles_across = vips_img.width // 512

    slide_label = int(classes[idx])
    with torch.no_grad():
        for index, (images, _, tile_ys, tile_xs) in tqdm(enumerate(loader)):
            images = images.to(DEVICE)
            pred_heatmap = torch.sigmoid(model(images))

            for i in range(images.shape[0]):
                tile_x = tile_xs[i].item()
                tile_y = tile_ys[i].item()

                image_np = images[i].cpu().permute(1, 2, 0).numpy()
                pred_heatmap_np = pred_heatmap[i, 0].cpu().numpy()
                centroids, num_cells = extract_cell_coordinates(
                    pred_heatmap_np, threshold=0.5, min_distance=25
                )
                if num_cells == 0:
                    continue
                centroids_merged_init = merge_close_centroids(
                    centroids, distance_threshold=35
                )

                centroids_merged = []
                for m, (cy, cx) in enumerate(centroids_merged_init):
                    if (
                        cx < margin
                        or cy < margin
                        or cx > (512 - margin)
                        or cy > (512 - margin)
                    ):
                        global_cx = tile_x + int(cx)
                        global_cy = tile_y + int(cy)
                        new_x = max(0, global_cx - 128)
                        new_y = max(0, global_cy - 128)
                        cropped = vips_img.crop(new_x, new_y, 256, 256)
                        patch = np.ndarray(
                            buffer=cropped.write_to_memory(),
                            dtype=np.uint8,
                            shape=[cropped.height, cropped.width, cropped.bands],
                        )
                        if patch.shape[2] == 4:
                            patch = patch[:, :, :3]
                        patch = cv2.GaussianBlur(patch, (3, 3), 0)
                        masked = binarize_HSV(patch)
                        dist = ndi.distance_transform_edt(masked)
                        markers = np.zeros(masked.shape, dtype=np.int32)
                        local_cx = global_cx - new_x
                        local_cy = global_cy - new_y
                        markers[local_cy, local_cx] = 1
                        seg_labels = np.array(watershed(-dist, markers, mask=masked))
                        labels, segmented_cells = filter_size(
                            seg_labels, 1, threshold=1000
                        )
                        bounding_boxes = calculate_bb(segmented_cells)
                        for bb in bounding_boxes:
                            x1, y1, x2, y2 = bb
                            all_records.append(
                                {
                                    "slide_id": file.stem,
                                    "label": slide_label,
                                    "x1": new_x + x1,
                                    "y1": new_y + y1,
                                    "x2": new_x + x2,
                                    "y2": new_y + y2,
                                }
                            )
                    else:
                        centroids_merged.append((cy, cx))

                if len(centroids_merged) == 0:
                    continue
                patch = cv2.GaussianBlur((image_np * 255).astype(np.uint8), (3, 3), 0)
                masked = binarize_HSV(patch)
                dist = ndi.distance_transform_edt(masked)
                markers = np.zeros(masked.shape, dtype=np.int32)
                for j, (y, x) in enumerate(centroids_merged, start=1):
                    markers[int(y), int(x)] = j

                seg_labels = np.array(watershed(-dist, markers, mask=masked))
                labels, segmented_cells = filter_size(
                    seg_labels, len(centroids_merged), threshold=1000
                )
                bounding_boxes = calculate_bb(segmented_cells)
                for bb in bounding_boxes:
                    x1, y1, x2, y2 = bb
                    all_records.append(
                        {
                            "slide_id": file.stem,
                            "label": slide_label,
                            "x1": tile_x + x1,
                            "y1": tile_y + y1,
                            "x2": tile_x + x2,
                            "y2": tile_y + y2,
                        }
                    )

    df = pd.DataFrame(all_records)
    out_path = Path("./bboxes") / f"{file.stem}_bboxes.parquet"
    out_path.parent.mkdir(exist_ok=True)
    df.to_parquet(out_path)
    print(f"{file.stem}: {len(df)} cells saved")
    vips_img = None

## Evaluation

### Get prediction

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from scripts.filters import *
from scripts.evaluate import *
from scripts.utils import *
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from pathlib import Path
import albumentations as A 
from albumentations.pytorch import ToTensorV2
from torchvision import transforms
import segmentation_models_pytorch as smp
import torch.nn as nn

data_split = "test"
IMAGE_DIR = f"./evaluation/{data_split}"
JSON_PATH = "./evaluation/annotations/gt_points.json"
CHECKPOINT = "./checkpoints/resnet18_sigma15_aug.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4
NUM_EPOCHS = 50
LR = 1e-4
VAL_SPLIT = 0.2
SIGMA = 15.0


class MicrogliaHeatmapDataset(Dataset):

    def __init__(self, image_dir, json_path, sigma=20.0, augment=False):
        self.image_dir = Path(image_dir)
        self.annotations = extract_points_from_json(json_path)
        self.file_names = [f.name for f in self.image_dir.glob("*.png")]
        self.sigma = sigma
        self.augment = augment
        self.train_aug = A.Compose(
            [
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.RandomBrightnessContrast(p=0.3),
                A.GaussNoise(p=0.2),
                A.GaussianBlur(blur_limit=3, p=0.1),
                A.Normalize(),
                ToTensorV2(),
            ]
        )
        self.val_aug = A.Compose(
            [
                A.Normalize(),
                ToTensorV2(),
            ]
        )
        self.heatmaps = {}
        for fname in self.file_names:
            with Image.open(self.image_dir / fname) as img:
                w, h = img.size
            pts = self.annotations.get(fname, [])
            self.heatmaps[fname] = make_gaussian_heatmap(
                (h, w), pts, sigma=self.sigma
            ).astype(np.float32)
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        fname = self.file_names[idx]
        with Image.open(self.image_dir / fname) as img:
            img = np.array(img.convert("RGB"))

        aug = self.train_aug if self.augment else self.val_aug
        result = aug(image=img, mask=self.heatmaps[fname])
        x = result["image"].float()
        y = result["mask"].unsqueeze(0).float()
        return x, y


full_dataset = MicrogliaHeatmapDataset(IMAGE_DIR, JSON_PATH, sigma=SIGMA)
model = smp.Unet().to(DEVICE)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5
)
checkpoint = torch.load(CHECKPOINT, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
inference_loader = DataLoader(full_dataset, batch_size=1, shuffle=False)
results = []

with torch.no_grad():
    for idx, (images, heatmaps_gt) in enumerate(inference_loader):
        images = images.to(DEVICE)
        pred_heatmap = torch.sigmoid(model(images))
        image_np = images[0].cpu().permute(1, 2, 0).numpy()
        pred_heatmap_np = pred_heatmap[0, 0].cpu().numpy()
        gt_heatmap_np = heatmaps_gt[0, 0].numpy()
        results.append(
            {
                "filename": full_dataset.file_names[idx],
                "image": image_np,
                "pred_heatmap": pred_heatmap_np,
                "gt_heatmap": gt_heatmap_np,
            }
        )

/home/sdog/anaconda3/envs/microglia-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Calculate Scores 

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from scripts.filters import *
from scripts.evaluate import *
from scripts.utils import *
import cv2
import pyvips
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from sklearn.cluster import DBSCAN
from scipy.ndimage import label, center_of_mass, maximum_filter


ground_truth = read_annotations("./evaluation/annotations/gt_bb.json")
ground_truth_pts = read_annotations_points("./evaluation/annotations/gt_points.json")
data_dir = f"./evaluation/{data_split}"
files = [f for f in Path(data_dir).glob("./*.png")]


def extract_cell_coordinates(heatmap, threshold=0.7, min_distance=20):
    """Given a heatmap, extract potential cell coordinates by finding local maxima above a threshold"""
    local_max = heatmap == maximum_filter(heatmap, size=min_distance)
    local_max = local_max & (heatmap > threshold)

    labeled, num_cells = label(local_max)
    centroids = center_of_mass(heatmap, labeled, range(1, num_cells + 1))

    return centroids, num_cells


def merge_close_centroids(centroids, distance_threshold=30):
    """Given potential cell centroids, merge those that are within the specified distance threshold into a single centroid using mean location"""
    if len(centroids) == 0:
        return []

    pts = np.array(centroids)
    labels = DBSCAN(eps=distance_threshold, min_samples=1).fit_predict(pts)

    merged = []
    for lab in np.unique(labels):
        cluster_pts = pts[labels == lab]
        merged_centroid = cluster_pts.mean(axis=0)
        merged.append(tuple(merged_centroid))

    return merged


def binarize_HSV(img_np):
    """Binarize image patch for branch extraction and bounding box calculation"""

    # HSV color segmentation:
    img_hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)

    lb = np.array([0, 0, 0])
    ub = np.array([15, 255, 255])
    mask = cv2.inRange(img_hsv, lb, ub)

    # morphology to connect branches
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)


preds_dict = {}
preds_dict_pts = {}
for idx, r in enumerate(results):

    pred_heatmap = r["pred_heatmap"]
    file = r["filename"]

    centroids, num_cells = extract_cell_coordinates(
        pred_heatmap, threshold=0.5, min_distance=25
    )
    centroids_merged = merge_close_centroids(centroids, distance_threshold=35)

    import matplotlib.pyplot as plt
    import pyvips

    img_file = pyvips.Image.new_from_file(
        f"./evaluation/test/{file}", access="sequential"
    )
    patch = np.ndarray(
        buffer=img_file.write_to_memory(),
        dtype=np.uint8,
        shape=[img_file.height, img_file.width, img_file.bands],
    )
    patch = cv2.GaussianBlur(patch, (3, 3), 0)
    masked = binarize_HSV(patch)

    edges = cv2.Canny(patch, 50, 150)

    dist = ndi.distance_transform_edt(masked)
    markers = np.zeros(masked.shape, dtype=np.int32)
    for i, (y, x) in enumerate(centroids_merged, start=1):
        markers[int(y), int(x)] = i

    labels = np.array(watershed(-dist, markers, mask=masked))
    labels, segmented_cells = filter_size(labels, len(centroids_merged), threshold=1000)
    bounding_boxes = calculate_bb(segmented_cells)
    final_bb = []
    for bb in bounding_boxes:
        x1, y1, x2, y2 = bb
        final_bb.append(bb)
    preds_dict[file] = final_bb
    preds_dict_pts[file] = centroids_merged

    # plot_predictions(
    #     img_name=file,
    #     gt_points=ground_truth_pts,
    #     gt_boxes_raw=ground_truth,
    #     pred_points=centroids_merged,
    #     pred_boxes=final_bb,
    #     dist_threshold=15,
    #     iou_threshold=0.3,
    #     it=None,
    #     data_set="test"
    # )


bulk_eval(
    ground_truth,
    preds_dict,
    iou_threshold=0.3,
    dist_threshold=0,
    points=False,
    plot=False,
)

bulk_eval(
    ground_truth_pts,
    preds_dict_pts,
    iou_threshold=0.3,
    dist_threshold=15,
    points=True,
    plot=False,
)

Bounding box evaluation
Evaluation output:
TP:37
FP:12
FN:9
Mean IOU:0.6507510439593678
F1-score: 0.7789473684210526
Pointwise evaluation
Evaluation output:
TP:41
FP:8
FN:5
Mean Pixelwise Euc. Dist.:4.087475232336415
F1-score: 0.8631578947368421


(41, 8, 5)